# DPO Preference Alignment with alignrl

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sacredvoid/alignrl/blob/main/notebooks/03_dpo_preference_alignment.ipynb)

This notebook demonstrates **Direct Preference Optimization (DPO)**, a technique for aligning language models with human preferences. DPO is a simpler alternative to RLHF (Reinforcement Learning from Human Feedback): instead of training a separate reward model and then doing PPO, DPO directly optimizes the policy using pairs of preferred and rejected responses. The result is a model that produces outputs humans tend to prefer.

## Setup

Install alignrl with training and Unsloth dependencies.

In [ ]:
!pip install alignrl[train,unsloth] -q

## How DPO Works (and Why It's Simpler Than RLHF)

Traditional RLHF has three stages:
1. Collect human preference data (chosen vs rejected responses)
2. Train a reward model on those preferences
3. Optimize the policy (your LLM) against the reward model using PPO

DPO collapses steps 2 and 3 into a single supervised learning objective. The key insight from Rafailov et al. (2023): the optimal policy under the RLHF objective has a closed-form relationship to the reward function. So instead of learning the reward and then optimizing against it, DPO directly optimizes the policy.

```
RLHF Pipeline (complex):            DPO Pipeline (simple):
+----------+    +--------+          +----------+
| Preference| -> | Reward | -> PPO   | Preference| -> Direct
| Data     |    | Model  |          | Data     |    Optimization
+----------+    +--------+          +----------+
  3 stages, 2 models                  1 stage, 1 model
```

### The DPO Loss

For each example, DPO computes:
- `log_prob(chosen)` under the current policy
- `log_prob(rejected)` under the current policy
- `log_prob(chosen)` under the reference model (frozen copy)
- `log_prob(rejected)` under the reference model (frozen copy)

The loss increases the probability of chosen responses relative to rejected ones, anchored by the reference model to prevent drift. The **beta** parameter controls this trade-off: higher beta = stay closer to the reference.

## Configuration

Key DPO parameters:
- **beta=0.1**: KL divergence penalty (standard default)
- **learning_rate=5e-7**: Very low LR since DPO is sensitive to overfitting
- **precompute_ref_log_probs=True**: Compute reference model log-probs once upfront (saves memory)

In [ ]:
from alignrl.dpo import DPOConfig, DPORunner

config = DPOConfig(
    model_name="Qwen/Qwen2.5-3B",
    output_dir="./outputs/dpo",
    dataset_name="HuggingFaceH4/ultrafeedback_binarized",
    dataset_split="train_prefs",
    dataset_size=500,
    max_steps=100,
    max_length=1024,
    beta=0.1,
    learning_rate=5e-7,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    precompute_ref_log_probs=True,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    report_to="none",
    logging_steps=10,
)

print(config.model_dump_json(indent=2))

## Data Exploration

UltraFeedback is a preference dataset where each example has a prompt, a **chosen** response (preferred by humans/AI), and a **rejected** response (lower quality). Let's inspect the data.

In [ ]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
print(f"Full dataset size: {len(ds):,} examples")
print(f"Columns: {ds.column_names}")
print()

# Look at a single example
example = ds[0]
print("=" * 60)
print("PROMPT:")
print("=" * 60)
# The chosen/rejected are lists of message dicts
# The prompt is everything except the last message
for msg in example["chosen"][:-1]:
    content = msg["content"][:200] + ("..." if len(msg["content"]) > 200 else "")
    print(f"  [{msg['role']}]: {content}")

print()
print("CHOSEN response (preferred):")
chosen_msg = example["chosen"][-1]
print(f"  {chosen_msg['content'][:300]}...")

print()
print("REJECTED response (lower quality):")
rejected_msg = example["rejected"][-1]
print(f"  {rejected_msg['content'][:300]}...")

Let's see how alignrl formats these examples for DPO training:

In [ ]:
from alignrl.dpo import format_ultrafeedback

formatted = format_ultrafeedback(ds[0])
print("Formatted for DPO:")
print(f"  prompt: {len(formatted['prompt'])} messages")
print(f"  chosen: {len(formatted['chosen'])} messages")
print(f"  rejected: {len(formatted['rejected'])} messages")
print()
print(f"  Chosen preview: {formatted['chosen'][0]['content'][:150]}...")
print(f"  Rejected preview: {formatted['rejected'][0]['content'][:150]}...")

## Training

The `DPORunner` handles loading the model with QLoRA, formatting the dataset, and running the TRL `DPOTrainer`. With `precompute_ref_log_probs=True`, the reference model log-probs are computed once before training starts, saving GPU memory during the training loop.

In [ ]:
runner = DPORunner(config)
result = runner.train()

print(f"Training complete!")
print(f"  Steps: {result.num_steps}")
print(f"  Final loss: {result.loss_history[-1]:.4f}")
print(f"  Adapter saved to: {result.output_dir}")

## Results: Training Loss and Preference Accuracy

For DPO, a decreasing loss means the model is increasingly assigning higher probability to chosen responses over rejected ones.

In [ ]:
import json
import matplotlib.pyplot as plt

with open("./outputs/dpo/train_result.json") as f:
    train_log = json.load(f)

plt.figure(figsize=(10, 5))
plt.plot(train_log["loss_history"], linewidth=2, color="#9333ea")
plt.xlabel("Logging Step")
plt.ylabel("DPO Loss")
plt.title("DPO Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss: {train_log['loss_history'][0]:.4f} -> {train_log['loss_history'][-1]:.4f}")
print(f"Metrics: {json.dumps(train_log.get('metrics', {}), indent=2)}")

## Inference: Comparing Base vs DPO

Let's compare the base model with the DPO-aligned model on prompts where preference alignment matters (helpfulness, safety, following instructions).

In [ ]:
from alignrl.inference import InferenceConfig, ModelServer, build_prompt

# Load base model
base_server = ModelServer(InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    backend="unsloth",
    max_tokens=256,
))
base_server.load()

# Load DPO-aligned model
dpo_server = ModelServer(InferenceConfig(
    model_name="Qwen/Qwen2.5-3B",
    adapter_path="./outputs/dpo/final",
    backend="unsloth",
    max_tokens=256,
))
dpo_server.load()

test_prompts = [
    "Explain quantum entanglement to a 10-year-old.",
    "What are the pros and cons of remote work?",
    "Write a short, professional email declining a meeting invitation.",
]

for prompt in test_prompts:
    messages = build_prompt(prompt, system="You are a helpful assistant.")
    base_resp = base_server.generate(messages)
    dpo_resp = dpo_server.generate(messages)

    print(f"Q: {prompt}")
    print(f"\n  [Base Model]:")
    print(f"  {base_resp[:300]}")
    print(f"\n  [DPO-Aligned]:")
    print(f"  {dpo_resp[:300]}")
    print("=" * 60)

## Save Adapter Weights

In [ ]:
from pathlib import Path

output_dir = Path("./outputs/dpo")

print("Saved files:")
for f in sorted(output_dir.rglob("*")):
    if f.is_file():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.relative_to(output_dir)} ({size_mb:.1f} MB)")

print(f"\nDPO adapter ready at: {output_dir / 'final'}")

## Next Steps

We've now completed all three training stages:
1. **SFT** (Notebook 01) - Instruction following
2. **GRPO** (Notebook 02) - Math reasoning with verifiable rewards
3. **DPO** (this notebook) - Human preference alignment

Next:
- **Notebook 04 (Evaluation)**: Benchmark all stages on standard benchmarks to measure improvement
- **Notebook 05 (Inference)**: Serve the best model with optimized backends